## 📂 Writing Files (Escritura de Datos desde Volúmenes)

Hasta ahora hemos visto cómo consultar archivos almacenados dentro de un **Volumen**, independientemente de si se encuentran en formato **CSV, JSON, Parquet**, entre otros.

Sin embargo, en un escenario real de Data Engineering, el objetivo no suele ser únicamente consultar estos archivos, sino también **persistir su información en una Delta Table**, aprovechando el formato de almacenamiento nativo y recomendado por Databricks, junto con todas las funcionalidades que ofrece Delta Lake.

Para realizar esta escritura, Databricks proporciona diferentes enfoques:

* 📝 Utilizando **CREATE TABLE AS SELECT (CTAs)**.
* ⚙️ Utilizando **USING + OPTIONS**.
* 🔄 Utilizando un enfoque híbrido **CTAs + READFILES**.
* 🐍 Utilizando **PySpark**.

En este capítulo nos centraremos en: **CREATE TABLE AS SELECT (CTAs)** y **USING + OPTIONS**, repasando además los conceptos estudiados anteriormente.



### 🚀 Punto de Inicio en Databricks

Antes de trabajar con Delta Lake necesitamos una sesión de Spark activa.

Spark será el motor encargado de:

* ✅ Leer datos
* ✅ Transformarlos
* ✅ Procesarlos de forma distribuida
* ✅ Persistirlos como Delta Tables

In [0]:
from pyspark.sql import SparkSession # Puerta de entrada para trabajar con spark <-- SIEMPRE DEBEMOS IMPORTAR LA LLAVE MAESTRA QUE INICIA TODO.
from pyspark.sql.functions import *  # Funciones propias del módulo SQL de Spark, para trabajar sobre Dataframes.
spark = SparkSession.builder.appName("13WritingFiles1").getOrCreate() 
"""
^          ^__________^        ^_________^                               ^
|                |                   |                                   | 
Variable   Constructor de Sesión   Nombre Aplicación       Evita conflicto del SparkSession"""

print("🚀 Spark Session iniciada correctamente")

### 🚀 CREATE TABLE AS SELECT (CTAs)

Como vimos en capítulos anteriores, **CTAS (Create Table As Select)** permite crear una nueva tabla a partir del resultado de una consulta SQL.

En este caso, la consulta puede realizarse sobre:

* 📂 Un archivo almacenado en un Volumen.
* 🗄️ Una Delta Table existente.

Durante el proceso, Databricks crea automáticamente la nueva tabla e **infiere el esquema** a partir del resultado de la consulta, convirtiendo a CTAS en una alternativa muy práctica cuando trabajamos con archivos almacenados en su estado original dentro de un Volumen.

> 📖 Si deseas profundizar en el funcionamiento de **CTAS**, puedes consultar el notebook correspondiente: **[08_CTAs](https://github.com/BrayanR03/Databricks-DE-2026/blob/main/08_CTAs.ipynb)**.


In [0]:
### DEFINIMOS LOS PATHS DE FUENTES ORIGEN DE DATOS (EXTERNOS E INTERNOS)

path_interno_source_data_csv = "/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/csv/"
path_interno_source_data_json = "/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/json/" 
#### 💡 PARA LOS EJEMPLOS SOLO UTILIZAREMOS LOS FORMATO CSV Y JSON

### DEFINIMOS LOS PATHS PARA LA ESCRITURA DE DATOS 

catalago = "catalog_databricks_2026_de"
esquema = "schema_databricks_2026_de"

# spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalago}")
print("Catalago creado correctamente")

# spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalago}.{esquema}")
print("Esquema creado correctamente")

#### ========= CSV =============

In [0]:
### ============== CTAs ============================= ###

### PASO A). PREPARAR QUERY SQL ( UTILIZAREMOS FORMA DE LECTURA READ_FILES )

display(spark.sql(f"""
                  
                  SELECT *
                  FROM read_files(
                      '{path_interno_source_data_csv}/sales_client_1.csv',
                      format => 'csv',
                      inferSchema => true,
                      header => true,
                      delimiter => ','
                  )

                  """))


### PASO B). ADJUNTAMOS QUERY A UN CTAs

spark.sql(f"""
          
        CREATE TABLE catalog_databricks_2026_de.schema_databricks_2026_de.ctas_sales_csv
        AS
        SELECT *
        FROM read_files(
            '{path_interno_source_data_csv}/sales_client_1.csv',
            format => 'csv',
            inferSchema => true,
            header => true,
            delimiter => ','
        )
         
          """)

print("Tabla creada correctamente a partir de un CTAs")

### PASO C). VERIFICAMOS CTAs
display(spark.sql("SELECT * FROM catalog_databricks_2026_de.schema_databricks_2026_de.ctas_sales_csv"))

#### ========= JSON =============

In [0]:
### ============== CTAs ============================= ###

### PASO A). PREPARAR QUERY SQL ( UTILIZAREMOS FORMA DE LECTURA READ_FILES )

display(spark.sql(f"""
                  
                  SELECT *
                  FROM read_files(
                      '{path_interno_source_data_json}/logs_oneline_1.json',
                      format => 'json'
                  )

                  """))


### PASO B). ADJUNTAMOS QUERY A UN CTAs

spark.sql(f"""
          
        CREATE TABLE catalog_databricks_2026_de.schema_databricks_2026_de.ctas_logs_json
        AS
        SELECT *
        FROM read_files(
            '{path_interno_source_data_json}/logs_oneline_1.json',
            format => 'json'
        )
         
          """)

print("Tabla creada correctamente a partir de un CTAs")

# ### PASO C). VERIFICAMOS CTAs
display(spark.sql("SELECT * FROM catalog_databricks_2026_de.schema_databricks_2026_de.ctas_logs_json"))

### 🚀 USING + OPTIONS

Otra de las alternativas para crear **Delta Tables** a partir de una fuente de datos consiste en utilizar las cláusulas **USING** y **OPTIONS**.

A diferencia de **CTAS**, donde la tabla se construye a partir del resultado de una consulta (`SELECT`), este enfoque permite definir explícitamente el formato de almacenamiento de la tabla y las opciones necesarias para leer la información proveniente de una fuente de datos.

⚠️ **Importante:** `USING + OPTIONS` siempre crea una **External Table**, sin importar el formato utilizado (CSV, PARQUET, DELTA, etc). Esto se debe a que la cláusula `OPTIONS(path = ...)` define un `LOCATION` externo, y Unity Catalog administra únicamente los metadatos de la tabla, mientras que los archivos físicos permanecen en la ubicación externa (S3, ADLS o GCS) previamente registrada mediante un **External Location**.

🗂️ Si el objetivo es crear una **Managed Table** (donde Databricks administra tanto los metadatos como los archivos físicos), no se debe usar `USING + OPTIONS` — este enfoque se aborda con otras alternativas como **CTAS** o `CREATE TABLE` + `COPY INTO`.

En esta demostración trabajaremos con **External Tables**, que es el único escenario posible mediante `USING + OPTIONS`.

#### 🌐 **External Table**

##### ========= CSV =============

In [0]:
### CREACIÓN DE DELTA TABLE

#### PASO 1). REALIZAR PASOS PREVIOS DE EXTERNAL TABLE

## https://github.com/BrayanR03/Databricks-DE-2026/blob/main/assets/pasos-previos-external-tables.md

#### PASO 2). REALIZAR USING + OPTIONS
spark.sql("""
          
          CREATE TABLE catalog_databricks_2026_de.schema_databricks_2026_de.using_options_table_csv
          (
              customer VARCHAR(50),
              orderid INT,
              total DECIMAL(10,2)
          )
          USING CSV
          OPTIONS
          (
              path = "s3://bucket-brayan-datasets/datasets-de-brayan/csv_datasets/sales_client_1.csv",
              header = "true",
              inferSchema = "true"
          )
          
          """)
print("Tabla creada correctamente")

#### PASO 3). VERIFICAR LOS DATOS
display(spark.sql("SELECT * FROM catalog_databricks_2026_de.schema_databricks_2026_de.using_options_table_csv"))

#### PASO 4). VERIFICAR METADATA DE TABLA CREADA
display(spark.sql("DESCRIBE EXTENDED catalog_databricks_2026_de.schema_databricks_2026_de.using_options_table_csv"))

##### ========= JSON =============

In [0]:
### CREACIÓN DE DELTA TABLE

#### PASO 1). REALIZAR PASOS PREVIOS DE EXTERNAL TABLE

## https://github.com/BrayanR03/Databricks-DE-2026/blob/main/assets/pasos-previos-external-tables.md

#### PASO 2). REALIZAR USING + OPTIONS
spark.sql("""
          
        CREATE TABLE catalog_databricks_2026_de.schema_databricks_2026_de.using_options_table_json (
            `timestamp`       TIMESTAMP,
            user_id           STRING,
            service           STRING,
            action            STRING,
            resource_id       STRING,
            status            STRING,
            response_time_ms  INT,
            ip_address        STRING,
            region            STRING,
            error_code        STRING,
            log_level         STRING
        )
        USING JSON
        OPTIONS
        (
            path = "s3://bucket-brayan-datasets/datasets-de-brayan/json_datasets/logs_oneline_1.json",
            inferSchema = "true"
        )
          
          """)
print("Tabla creada correctamente")

# #### PASO 3). VERIFICAR LOS DATOS
display(spark.sql("SELECT * FROM catalog_databricks_2026_de.schema_databricks_2026_de.using_options_table_json"))

# #### PASO 4). VERIFICAR METADATA DE TABLA CREADA
display(spark.sql("DESCRIBE EXTENDED catalog_databricks_2026_de.schema_databricks_2026_de.using_options_table_json"))